<div align="center">
    <span style="font-size:30px">
        <strong>
            <!-- Símbolo de Python -->
            <img
                src="https://cdn3.emoji.gg/emojis/1887_python.png"
                style="margin-bottom:-5px"
                width="30px" 
                height="30px"
            >
            <!-- Título -->
            Python para Geólogos
            <!-- Versión -->
            <img 
                src="https://img.shields.io/github/release/kevinalexandr19/manual-python-geologia.svg?style=flat&label=&color=blue"
                style="margin-bottom:-2px" 
                width="40px"
            >
        </strong>
    </span>
    <br>
    <span>
        <!-- Github del proyecto -->
        <a href="https://github.com/kevinalexandr19/manual-python-geologia" target="_blank">
            <img src="https://img.shields.io/github/stars/kevinalexandr19/manual-python-geologia.svg?style=social&label=Github Repo">
        </a>
        &nbsp;&nbsp;
        <!-- Licencia -->
        <img src="https://img.shields.io/github/license/kevinalexandr19/manual-python-geologia.svg?color=forestgreen">
        &nbsp;&nbsp;
        <!-- Release date -->
        <img src="https://img.shields.io/github/release-date/kevinalexandr19/manual-python-geologia?color=gold">
    </span>
    <br>
    <span>
        <!-- Perfil de LinkedIn -->
        <a target="_blank" href="https://www.linkedin.com/in/kevin-alexander-gomez/">
            <img src="https://img.shields.io/badge/-Kevin Alexander Gomez-5eba00?style=social&logo=linkedin">
        </a>
        &nbsp;&nbsp;
        <!-- Perfil de Github -->
        <a target="_blank" href="https://github.com/kevinalexandr19">
            <img src="https://img.shields.io/github/followers/kevinalexandr19.svg?style=social&label=kevinalexandr19&maxAge=2592000">
        </a>
    </span>
    <br>
</div>

***

<span style="color:lightgreen; font-size:25px">**PG303 - Análisis geoespacial** </span>

Bienvenido al curso!!!

Vamos a revisar algunas aplicaciones del <span style="color:gold">análisis geoespacial</span> usando código en Python. <br>
Es necesario que tengas un conocimiento previo en programación con Python, estadística y sistemas de información geográfica.


<span style="color:gold; font-size:20px">**Shapely**</span>

***
- [¿Qué es Shapely?](#parte-1)
- [Geometría de puntos con Point](#parte-2)
- [Geometría de líneas con LineString](#parte-3)
- [Geometría de polígonos con Polygon](#parte-4)
- [Manipulación de objetos geométricos](#parte-5)
- [Análisis de objetos geométricos](#parte-6)

***

<a id="parte-1"></a>

### <span style="color:lightgreen">**¿Qué es Shapely?**</span>
***

El análisis espacial es un componente importante para resolver problemas en geociencias. Algunas tareas comunes son medir perímetros, áreas y distancias o seleccionar elementos según su relación espacial.

<span style="color:gold">Shapely</span> trabaja con la **forma** de objetos vectoriales como puntos, líneas y polígonos. Un punto puede representar una muestra, una línea el trazo de una falla y un polígono una unidad cartografiada. Un ráster, como un mapa de elevación, guarda un valor por celda de una malla; lo veremos en el cuaderno de Rioxarray.

Usaremos coordenadas planas de un levantamiento ficticio en **metros**. Shapely opera sobre el plano X–Y e ignora Z en el análisis geométrico. Para combinar capas reales, primero hay que comprobar que compartan un sistema de coordenadas adecuado. JupyterLite carga Shapely desde el entorno del curso. En Python local puedes instalarlo con `python -m pip install "shapely>=2,<3" pandas matplotlib`.

In [ ]:
import sys

if sys.platform == "emscripten":
    from pyodide_js import loadPackage

    await loadPackage(["shapely", "pandas", "matplotlib"])

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from shapely import dwithin, make_valid, unary_union
from shapely.geometry import LineString, MultiPoint, Point, Polygon
from shapely.validation import explain_validity

<a id="parte-2"></a>

### <span style="color:lightgreen">**Geometría de puntos con Point**</span>
***

`Point` recibe un par `(x, y)`. Prueba calcular a mano la distancia entre A y B antes de leer la salida. `MultiPoint` conserva varios puntos en un solo objeto geométrico.

In [ ]:
A = Point(0, 0)
B = Point(1, 1)
muestras = MultiPoint([(0, 0), (1, 1), (0, 1), (1, 0), (2, 2), (2, 1)])

print(f"A está en ({A.x:.0f}, {A.y:.0f}) m")
print(f"Distancia A–B: {A.distance(B):.3f} m")
print(f"Puntos en la colección: {len(muestras.geoms)}")

<a id="parte-3"></a>

### <span style="color:lightgreen">**Geometría de líneas con LineString**</span>
***

`LineString` conecta los vértices en el orden de entrada. `length` suma la longitud de los tramos. En el primer ejemplo la línea tiene los mismos extremos que A y B, de modo que puedes comparar las dos medidas.

In [ ]:
linea = LineString([(0, 0), (1, 1)])
print(f"Longitud de la línea: {linea.length:.3f} m")

<a id="parte-4"></a>

### <span style="color:lightgreen">**Geometría de polígonos con Polygon**</span>
***

`Polygon` usa una secuencia de vértices para definir el contorno. Este cuadrado tiene lados de **5 m**. Comprueba que su área sea 25 m² y su perímetro 20 m. `bounds` devuelve cuatro números en el orden `xmin, ymin, xmax, ymax`.

In [ ]:
poligono = Polygon([(0, 0), (0, 5), (5, 5), (5, 0)])
xmin, ymin, xmax, ymax = poligono.bounds
centroide = poligono.centroid
print(f"Área: {poligono.area:.0f} m² | Perímetro: {poligono.length:.0f} m")
print(f"Extensión: {poligono.bounds} | Centroide: ({centroide.x}, {centroide.y}) m")

<a id="parte-5"></a>

### <span style="color:lightgreen">**Manipulación de objetos geométricos**</span>
***

Conservamos las dos líneas del ejemplo original. Recorre las coordenadas y anticipa qué tramo tienen en común. `difference` conserva la parte de A que queda fuera de B. `intersection` selecciona lo compartido. `symmetric_difference` conserva los tramos exclusivos de A y B, y `union` reúne ambas líneas. `unary_union` trabaja sobre una secuencia de geometrías.

In [ ]:
trazo_a = LineString([(0, 0), (1, 1), (1, 2), (2, 2)])
trazo_b = LineString([(0, 0), (1, 1), (2, 1), (2, 2)])
operaciones = {
    "A − B": trazo_a.difference(trazo_b),
    "A ∩ B": trazo_a.intersection(trazo_b),
    "Diferencia simétrica": trazo_a.symmetric_difference(trazo_b),
    "A ∪ B": trazo_a.union(trazo_b),
    "unary_union": unary_union([trazo_a, trazo_b]),
}
pd.DataFrame(
    [(nombre, geom.geom_type, geom.length, geom.is_empty)
     for nombre, geom in operaciones.items()],
    columns=["Operación", "Tipo", "Longitud (m)", "Vacío"],
).round(3)

La salida puede ser una sola geometría o una colección. Aquí A y B comparten una línea y un punto, por lo que se obtiene una `GeometryCollection` y se pueden recorrer sus partes con `geoms`. Dos líneas que se cruzan una sola vez devuelven un `Point`. Si no se tocan, la intersección es vacía y se consulta con `is_empty`.

In [ ]:
compartido = operaciones["A ∩ B"]
cruce = linea.intersection(LineString([(0, 1), (1, 0)]))
sin_contacto = linea.intersection(LineString([(2, 0), (3, 0)]))
print([parte.geom_type for parte in compartido.geoms])
print(cruce.geom_type, cruce.wkt, "| Vacío:", sin_contacto.is_empty)

fig, ax = plt.subplots(figsize=(5.2, 4.2), layout="constrained")
ax.plot(*trazo_a.xy, color="#0072B2", lw=3, label="Línea A")
ax.plot(*trazo_b.xy, color="#B84D12", lw=3, label="Línea B")
for parte in compartido.geoms:
    if parte.geom_type == "Point":
        ax.scatter(parte.x, parte.y, color="#087E73", s=80, zorder=4)
    else:
        ax.plot(*parte.xy, color="#087E73", lw=4, ls="--", label="Tramo compartido")
ax.annotate("Punto compartido", xy=(2, 2), xytext=(.62, 2.25),
            arrowprops={"arrowstyle": "->", "color": "#087E73"})
ax.set(xlim=(-.2, 2.4), ylim=(-.2, 2.55), aspect="equal", title="Intersección de dos líneas",
       xlabel="X (m)", ylabel="Y (m)")
ax.legend(loc="lower right", fontsize=8)
ax.grid(alpha=.2)
plt.show()

`buffer` dibuja una franja a una distancia dada. En un trazo de falla, el ancho podría representar un criterio de proximidad. `convex_hull` devuelve la envolvente convexa de la colección de muestras. Observa cuánto espacio entre puntos queda incluido dentro de la envolvente.

In [ ]:
franja = trazo_a.buffer(.3)
envolvente = muestras.convex_hull
print(f"Área de la franja de 0.3 m: {franja.area:.2f} m²")
print(f"Área de la envolvente: {envolvente.area:.2f} m²")

<a id="parte-6"></a>

### <span style="color:lightgreen">**Análisis de objetos geométricos**</span>
***

Antes de ejecutar la siguiente celda, decide si aceptarías una muestra exactamente sobre el límite del polígono. Para estos puntos `contains` prueba pertenencia al interior y `covers` incluye el borde. `intersects` detecta cualquier contacto. `dwithin` comprueba si la distancia es menor o igual al umbral.

In [ ]:
posiciones = {"Interior": Point(2, 2), "Borde": Point(5, 2), "Exterior": Point(6, 2)}
resultados = pd.DataFrame(
    [(nombre, poligono.contains(p), poligono.covers(p), poligono.intersects(p),
      dwithin(poligono, p, 1))
     for nombre, p in posiciones.items()],
    columns=["Posición", "contains", "covers", "intersects", "A ≤ 1 m"],
).set_index("Posición")
resultados

In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 4.2), layout="constrained")
ax.fill(*poligono.exterior.xy, color="#BFE4DC", edgecolor="#087E73", lw=2)
for (nombre, p), color, marca in zip(posiciones.items(),
                                   ["#0072B2", "#B84D12", "#70428C"], ["o", "s", "D"]):
    ax.scatter(p.x, p.y, color=color, marker=marca, s=62, zorder=3, label=nombre)
ax.set(xlim=(-.5, 6.7), ylim=(-.5, 5.7), aspect="equal",
       title="Tres muestras frente al mismo límite", xlabel="X (m)", ylabel="Y (m)")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1))
ax.grid(alpha=.2)
plt.show()

`crosses` comprueba si dos líneas se cruzan en su interior. Antes de hacer operaciones con polígonos reales, revisa `is_valid`. En el siguiente ejemplo el contorno se cruza a sí mismo. `make_valid` puede cambiar el tipo de salida, así que compara el resultado con la geometría que esperabas.

In [ ]:
cruzada = LineString([(0, 1), (1, 0)])
lazo = Polygon([(0, 0), (0, 2), (2, 0), (2, 2)])
reparado = make_valid(lazo)
print("¿Se cruzan las líneas?", linea.crosses(cruzada))
print("Polígono original:", lazo.is_valid, explain_validity(lazo))
print("Resultado de la reparación:", reparado.geom_type, "| válido:", reparado.is_valid)

**Comprueba tu lectura.** Mueve el punto del borde 0.1 m hacia afuera. Anticipa cuáles columnas de la tabla cambian. Si tus coordenadas reales estuvieran en grados, ¿qué paso agregarías antes de usar un `buffer` de 0.3 m?

[Manual de Shapely](https://shapely.readthedocs.io/en/2.0.6/manual.html)